# Dán nhãn bình luận bằng Gemini - Kaggle

Đây là bản Kaggle của notebook Gemini trên Colab.

Chuẩn bị:

1. Tạo Kaggle Dataset chứa final.csv và thư mục Annotation có prompt.
2. Add Dataset đó vào notebook Kaggle.
3. Tạo Kaggle Secret tên GOOGLE_API_KEY.
4. Bật Internet trong Notebook options.
5. Chạy các cell theo thứ tự.

Notebook tự tìm final.csv hoặc comments_cleaned_to_5_4.csv trong /kaggle/input,
và prompt_dan_nhan_llm.md trong /kaggle/input.

LLM chỉ tiền dán nhãn. Kết quả needs_review=true vẫn phải được người kiểm
duyệt xác nhận.


In [ ]:
# Cài thư viện vào kernel Kaggle hiện tại
%pip install -q -U google-generativeai pandas


In [ ]:
from pathlib import Path
import hashlib
import html
import json
import os
import re
import time

import pandas as pd
import google.generativeai as genai

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

INPUT_PATH = None
PROMPT_PATH = None

INPUT_FILE_NAMES = ['final.csv', 'comments_cleaned_to_5_4.csv']
PROMPT_FILE_NAME = 'prompt_dan_nhan_llm.md'

OUTPUT_DIR = Path('/kaggle/working/gemini_annotation')
OUTPUT_PATH = OUTPUT_DIR / 'annotation_gemini.csv'
CHECKPOINT_PATH = OUTPUT_DIR / 'annotation_gemini_checkpoint.csv'

MODEL_NAME = 'gemini-2.0-flash'
PROMPT_VERSION = 'ds108-compact-v1'
REQUEST_INTERVAL = 4.1
MAX_ROWS = None
START_ROW_ID = None
RETRY_REVIEW_ROWS = False

from IPython.display import display


In [ ]:
# Tìm input và prompt trong Kaggle Dataset

def find_file_in_kaggle(file_names):
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    matches = []
    for root in roots:
        if root.exists():
            for file_name in file_names:
                matches.extend(root.rglob(file_name))
    return sorted(set(path.resolve() for path in matches))

if INPUT_PATH is None:
    input_matches = find_file_in_kaggle(INPUT_FILE_NAMES)
    if not input_matches:
        raise FileNotFoundError(
            'Không tìm thấy final.csv hoặc comments_cleaned_to_5_4.csv trong /kaggle/input.'
        )
    preferred = [path for path in input_matches if path.name == 'final.csv']
    INPUT_PATH = preferred[0] if preferred else input_matches[0]
else:
    INPUT_PATH = Path(INPUT_PATH)

if PROMPT_PATH is None:
    prompt_matches = find_file_in_kaggle([PROMPT_FILE_NAME])
    if not prompt_matches:
        raise FileNotFoundError(
            'Không tìm thấy prompt_dan_nhan_llm.md trong /kaggle/input.'
        )
    PROMPT_PATH = prompt_matches[0]
else:
    PROMPT_PATH = Path(PROMPT_PATH)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy input: {INPUT_PATH}')
if not PROMPT_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy prompt: {PROMPT_PATH}')

print(f'Input: {INPUT_PATH}')
print(f'Prompt: {PROMPT_PATH}')
print(f'Output: {OUTPUT_PATH}')


In [ ]:
# Đọc prompt và tạo các hàm xử lý văn bản

PROMPT_TEMPLATE = PROMPT_PATH.read_text(encoding='utf-8')

def clean_text(value):
    text = '' if pd.isna(value) else str(value)
    text = html.unescape(text)
    return re.sub(r'\s+', ' ', text).strip()

def render_prompt(comment):
    return PROMPT_TEMPLATE.replace('{comment}', clean_text(comment))


In [ ]:
# Lấy API key từ Kaggle Secret hoặc biến môi trường

def get_google_api_key():
    if UserSecretsClient is not None:
        try:
            secret = UserSecretsClient().get_secret('GOOGLE_API_KEY')
            if secret:
                return secret
        except Exception:
            pass

    secret = os.environ.get('GOOGLE_API_KEY')
    if secret:
        return secret

    raise RuntimeError(
        'Chưa tìm thấy GOOGLE_API_KEY. Hãy tạo Kaggle Secret có tên GOOGLE_API_KEY.'
    )

api_key = get_google_api_key()
genai.configure(api_key=api_key)
print('Đã cấu hình Gemini; API key không được in ra.')


In [ ]:
ALLOWED_TOXICITY = {'TOXIC', 'NON-TOXIC'}
ALLOWED_TYPES = {'HATE', 'OFFENSIVE'}
ALLOWED_EXPRESSIONS = {'EXPLICIT', 'IMPLICIT', 'REPORTING'}

def validate_annotation(annotation):
    required = {'toxicity', 'hate_offensive', 'expression_type', 'evidence',
                'needs_review', 'review_reason'}
    if not required.issubset(annotation):
        raise ValueError(f'Thiếu trường: {sorted(required - set(annotation))}')

    toxicity = annotation['toxicity']
    if toxicity not in ALLOWED_TOXICITY and not (toxicity is None and annotation['needs_review']):
        raise ValueError(f'toxicity không hợp lệ: {toxicity}')

    if toxicity == 'NON-TOXIC':
        if annotation['hate_offensive'] is not None or annotation['expression_type'] is not None:
            raise ValueError('NON-TOXIC phải có nhãn phụ là null.')
    elif toxicity == 'TOXIC':
        if annotation['hate_offensive'] not in ALLOWED_TYPES:
            raise ValueError('TOXIC phải có HATE hoặc OFFENSIVE.')
        if annotation['expression_type'] not in ALLOWED_EXPRESSIONS:
            raise ValueError('TOXIC phải có EXPLICIT, IMPLICIT hoặc REPORTING.')
    else:
        if annotation['hate_offensive'] is not None or annotation['expression_type'] is not None:
            raise ValueError('Kết quả review phải có nhãn phụ là null.')
        if not annotation['needs_review']:
            raise ValueError('Kết quả null phải được đánh dấu needs_review=true.')

    if not isinstance(annotation['needs_review'], bool):
        raise ValueError('needs_review phải là boolean.')
    return annotation

def parse_json_response(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', cleaned, flags=re.IGNORECASE)
    match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError('LLM không trả về JSON object.')
    return validate_annotation(json.loads(match.group(0)))


In [ ]:
# Khởi tạo model và hàm gọi Gemini có giới hạn tốc độ

model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config={
        'temperature': 0.0,
        'top_p': 1.0,
        'max_output_tokens': 600,
    },
)

last_request_time = 0.0

def annotate(comment):
    global last_request_time
    elapsed = time.time() - last_request_time
    if elapsed < REQUEST_INTERVAL:
        time.sleep(REQUEST_INTERVAL - elapsed)

    response = model.generate_content(render_prompt(comment))
    last_request_time = time.time()
    raw_response = getattr(response, 'text', '') or ''
    if not raw_response.strip():
        raise ValueError('Gemini trả về response rỗng hoặc bị chặn.')

    cleaned = raw_response.strip()
    fence = chr(96) * 3
    if cleaned.startswith(fence):
        cleaned = cleaned[len(fence):].strip()
        if cleaned.lower().startswith('json'):
            cleaned = cleaned[4:].strip()
        if cleaned.endswith(fence):
            cleaned = cleaned[:-len(fence)].strip()

    match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError('Gemini không trả về JSON object.')
    return validate_annotation(json.loads(match.group(0))), raw_response


In [ ]:
# Đọc dữ liệu và tạo batch cần xử lý

df = pd.read_csv(
    INPUT_PATH,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
)

required_columns = {'comment'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

if 'row_id' not in df.columns:
    df.insert(0, 'row_id', range(len(df)))

df['comment'] = df['comment'].map(clean_text)
batch = df.copy()

if START_ROW_ID is not None:
    batch = batch[batch['row_id'].astype(str) >= str(START_ROW_ID)]
if MAX_ROWS is not None:
    batch = batch.head(MAX_ROWS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
completed_records = []
if CHECKPOINT_PATH.exists():
    checkpoint = pd.read_csv(
        CHECKPOINT_PATH,
        encoding='utf-8-sig',
        dtype='string',
        keep_default_na=False,
    )
    completed_records = checkpoint.to_dict('records')
    completed_ids = set(checkpoint['row_id'].astype(str))
    if not RETRY_REVIEW_ROWS:
        batch = batch[~batch['row_id'].astype(str).isin(completed_ids)]

print(f'Tổng dữ liệu: {len(df):,}')
print(f'Số dòng trong batch: {len(batch):,}')
print(f'Đã có trong checkpoint: {len(completed_records):,}')


In [ ]:
# Gán nhãn từng comment và lưu checkpoint sau mỗi dòng

records = list(completed_records)

for _, row in batch.iterrows():
    comment = clean_text(row.get('comment', ''))
    input_hash = hashlib.sha256(comment.encode('utf-8')).hexdigest()

    try:
        annotation, raw_response = annotate(comment)
    except Exception as exc:
        annotation = {
            'toxicity': None,
            'hate_offensive': None,
            'expression_type': None,
            'evidence': '',
            'needs_review': True,
            'review_reason': f'LLM_OR_PARSE_ERROR: {type(exc).__name__}',
        }
        raw_response = str(exc)

    records.append({
        'row_id': row['row_id'],
        'platform': row.get('platform', ''),
        'comment': comment,
        'input_hash': input_hash,
        **annotation,
        'raw_response': raw_response,
        'model': MODEL_NAME,
        'prompt_version': PROMPT_VERSION,
    })

    pd.DataFrame(records).to_csv(
        CHECKPOINT_PATH,
        index=False,
        encoding='utf-8-sig',
    )

result = pd.DataFrame(records)
result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f'Đã lưu checkpoint: {CHECKPOINT_PATH}')
print(f'Đã lưu kết quả: {OUTPUT_PATH}')
print(f'Số bản ghi: {len(result):,}')


In [ ]:
# Kiểm tra kết quả

result = pd.read_csv(
    OUTPUT_PATH,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
)

print(result['toxicity'].value_counts(dropna=False))
print('needs_review:', result['needs_review'].value_counts(dropna=False).to_dict())

review_queue = result[
    result['needs_review'].astype(str).str.lower().eq('true')
]
print(f'Review queue: {len(review_queue):,}')
print(f'Output path: {OUTPUT_PATH}')
display(result.head())


In [ ]:
# Tạo file review queue và liên kết tải file

from IPython.display import FileLink, display

review_queue_path = OUTPUT_DIR / 'annotation_gemini_review_queue.csv'
review_queue.to_csv(
    review_queue_path,
    index=False,
    encoding='utf-8-sig',
)

display(FileLink(str(OUTPUT_PATH)))
display(FileLink(str(review_queue_path)))
print('Bấm vào liên kết để tải file về máy.')
